##Parameter Setup
####Central configuration notebook - defines all reusable parameters including ADLS paths, Key Vault secret scope, catalog and schema names; imported via %run across all notebooks

In [0]:
dbutils.widgets.text("load_type", "")
dbutils.widgets.text("processing_year", "")

In [0]:
load_type = dbutils.widgets.get("load_type")
processing_year = dbutils.widgets.get("processing_year")

if load_type == "static":
  load = "ref/config"
else:
  load = "config"

In [0]:
# Fetch secrets
client_id      = dbutils.secrets.get(scope="e-commerce-secret-scope", key="clientID")
client_secret  = dbutils.secrets.get(scope="e-commerce-secret-scope", key="clientSecret")
tenant_id      = dbutils.secrets.get(scope="e-commerce-secret-scope", key="tenantId")
storage_name   = dbutils.secrets.get(scope="e-commerce-secret-scope", key="storageAccountName")
container_name = dbutils.secrets.get(scope="e-commerce-secret-scope", key="containerName")

In [0]:
# Set Spark config
spark.conf.set(f"fs.azure.account.auth.type.{storage_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

In [0]:
# Storing the storage account path
#Need to set storage base path for reference and transactional
storage_base_path = f"abfss://{container_name}@{storage_name}.dfs.core.windows.net"

In [0]:
# Reading the config folder
config_file = spark.read.option("multiLine", "true").json(f"{storage_base_path}/{load}/config.json")

if load_type != "static":
    last_run_ops_config_path = f"{storage_base_path}/{load}/last_run_ops_daily.json"  # ← path string
    last_run_ops_config = spark.read.option("multiLine", "true").json(last_run_ops_config_path)  # ← DataFrame for reading

In [0]:
# Setting up the inbound folder path
inbound_folder = config_file.select("inbound_folder").first()[0]
inbound_path = f"{storage_base_path}/"+inbound_folder

In [0]:
# Setting up the raw folder path
raw_folder = config_file.select("raw_folder").first()[0]
raw_path = f"{storage_base_path}/"+raw_folder

In [0]:
# Setting up the struct folder path
struct_folder = config_file.select("struct_folder").first()[0]
struct_path = f"{storage_base_path}/"+struct_folder

In [0]:
# Setting up the prep folder path
prep_folder = config_file.select("prep_folder").first()[0]
prep_path = f"{storage_base_path}/"+prep_folder

In [0]:
LOCAL_PDF_PATH = f"/Workspace/{container_name}/reports/ecom_dashboard_{processing_year}.pdf"
ADLS_PDF_PATH  = f"{storage_base_path}/reports/ecom_dashboard_{processing_year}.pdf"

In [0]:
print("Parameter setup completed")